# Stage 01 Stable Geometry, Energy, and Solver Review

这个 notebook 只读展示 Stage 01 在稳定几何上的关键证据：

- `triangle_area >= 0.10`、`cond(J) <= 40.0` 为什么能把解析链路稳定下来
- 绝对能量为什么可能帮助消除 `TDOA-only` 的多解歧义
- 纯 MLP、带物理损失的 MLP、可微解析层 solver 三条路线在同一份 `L1 stable` 数据上的对照


In [ ]:
from pathlib import Path
import csv
import json
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
for parent in [ROOT] + list(ROOT.parents):
    if (parent / 'python_impl').exists() and (parent / 'README.md').exists():
        ROOT = parent
        break
SRC_DIR = ROOT / 'python_impl' / 'experiments' / 'hypothesis_validation' / '01_source_localization_anechoic_2d' / 'src'
if str(ROOT / 'python_impl') not in sys.path:
    sys.path.insert(0, str(ROOT / 'python_impl'))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from python_scripts.hypothesis_validation_common import estimate_source_position_from_tdoa, estimate_tdoa_from_gcc_triplet
from stable_geometry_pipeline import (
    find_tdoa_room_solutions,
    load_geometry_audit_bundle,
    load_stage01_candidate_prediction_bundle,
    load_stable_localization_summary,
)

STEP_DIR = ROOT / 'python_impl' / 'experiments' / 'hypothesis_validation' / '01_source_localization_anechoic_2d'
AUDIT_DIR = STEP_DIR / 'results' / 'geometry_stability_audit'
H5_PATH = STEP_DIR / 'data' / 'source_localization_anechoic_2d_l1_stable.h5'
RESULT_DIR = STEP_DIR / 'results' / 'L1_stable'
MLP_RESULT_DIR = STEP_DIR / 'results' / 'L1_stable_energy_canonical'
ABS_RESULT_DIR = STEP_DIR / 'results' / 'L1_stable_energy_abs_canonical_v1'
PHYS_RESULT_DIR = STEP_DIR / 'results' / 'L1_stable_phys_baseline_v2'
SOLVER_RESULT_DIR = STEP_DIR / 'results' / 'L1_stable_solver_tdoa_energy_v4'
SAMPLE_INDEX = -1

print('ROOT =', ROOT)
print('AUDIT_DIR =', AUDIT_DIR)
print('H5_PATH =', H5_PATH)
print('RESULT_DIR =', RESULT_DIR)
print('MLP_RESULT_DIR =', MLP_RESULT_DIR)
print('ABS_RESULT_DIR =', ABS_RESULT_DIR)
print('PHYS_RESULT_DIR =', PHYS_RESULT_DIR)
print('SOLVER_RESULT_DIR =', SOLVER_RESULT_DIR)


In [ ]:
audit = load_geometry_audit_bundle(AUDIT_DIR)
audit_summary = audit['summary']
selected = audit_summary['selected_thresholds']
threshold_df = pd.DataFrame(audit['threshold_grid']).apply(pd.to_numeric, errors='ignore')
sample_df = pd.DataFrame(audit['sample_metrics']).apply(pd.to_numeric, errors='ignore')

display(pd.Series(selected, name='selected_thresholds'))
baseline_table = pd.DataFrame([
    {
        'mode': 'raw_W1_gcc_phat',
        'iid_median_m': audit_summary['runs']['W1']['gcc_phat_iid_test']['median_m'],
        'iid_p90_m': audit_summary['runs']['W1']['gcc_phat_iid_test']['p90_m'],
        'geom_median_m': audit_summary['runs']['W1']['gcc_phat_geom_test']['median_m'],
        'geom_p90_m': audit_summary['runs']['W1']['gcc_phat_geom_test']['p90_m'],
    },
    {
        'mode': 'stable_filtered_gcc_phat',
        'iid_median_m': selected['gcc_phat_iid_test_median_m'],
        'iid_p90_m': selected['gcc_phat_iid_test_p90_m'],
        'geom_median_m': selected['gcc_phat_geom_test_median_m'],
        'geom_p90_m': selected['gcc_phat_geom_test_p90_m'],
    },
])
display(baseline_table)

fig, axes = plt.subplots(2, 2, figsize=(13.5, 10.0), constrained_layout=True)
inside = sample_df['inside_convex_hull'].to_numpy() > 0.5

axes[0, 0].scatter(sample_df['triangle_area'], sample_df['gcc_phat_position_error_m'], s=4, alpha=0.10)
axes[0, 0].set_xlabel('triangle_area')
axes[0, 0].set_ylabel('position_error_m')
axes[0, 0].set_title('GCC-PHAT error vs triangle area')
axes[0, 0].grid(True, alpha=0.25)

axes[0, 1].scatter(sample_df.loc[~inside, 'jacobian_condition'], sample_df.loc[~inside, 'gcc_phat_position_error_m'], s=4, alpha=0.08, label='outside')
axes[0, 1].scatter(sample_df.loc[inside, 'jacobian_condition'], sample_df.loc[inside, 'gcc_phat_position_error_m'], s=4, alpha=0.16, label='inside')
axes[0, 1].set_xscale('log')
axes[0, 1].set_xlabel('cond(J)')
axes[0, 1].set_ylabel('position_error_m')
axes[0, 1].set_title('GCC-PHAT error vs condition number')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.25)

axes[1, 0].scatter(sample_df['centroid_source_dist_norm'], sample_df['gcc_phat_position_error_m'], s=4, alpha=0.10)
axes[1, 0].set_xlabel('normalized source radius')
axes[1, 0].set_ylabel('position_error_m')
axes[1, 0].set_title('Error vs normalized source radius')
axes[1, 0].grid(True, alpha=0.25)

pivot = threshold_df.pivot(index='area_min', columns='cond_max', values='retention_overall')
heat = axes[1, 1].imshow(pivot.to_numpy(), origin='lower', aspect='auto')
axes[1, 1].set_xticks(np.arange(pivot.shape[1]), labels=[str(int(v)) for v in pivot.columns])
axes[1, 1].set_yticks(np.arange(pivot.shape[0]), labels=[f'{v:.2f}' for v in pivot.index])
axes[1, 1].set_xlabel('cond_max')
axes[1, 1].set_ylabel('area_min')
axes[1, 1].set_title('Retention heatmap')
fig.colorbar(heat, ax=axes[1, 1], fraction=0.046, pad=0.04)
plt.show()


In [ ]:
stable_summary = load_stable_localization_summary(RESULT_DIR)
with h5py.File(H5_PATH, 'r') as h5:
    ref_positions = np.asarray(h5['raw/ref_positions'], dtype=np.float32)
    source_position = np.asarray(h5['raw/source_position'], dtype=np.float32)
    x_ref = np.asarray(h5['raw/x_ref'], dtype=np.float32)
    gcc_phat = np.asarray(h5['raw/gcc_phat'], dtype=np.float32)
    true_tdoa = np.asarray(h5['raw/true_tdoa'], dtype=np.float32)
    geom_metrics = np.asarray(h5['raw/geometry_metrics'], dtype=np.float32)
    metric_names = json.loads(h5.attrs['geometry_metric_names_json'])
    cfg = json.loads(h5.attrs['config_json'])
geom_name_to_idx = {name: i for i, name in enumerate(metric_names)}

comparison_rows = [
    {
        'method': 'oracle_from_true_tdoa',
        'iid_median_m': selected['oracle_iid_test_median_m'],
        'iid_p90_m': selected['oracle_iid_test_p90_m'],
        'geom_median_m': selected['oracle_geom_test_median_m'],
        'geom_p90_m': selected['oracle_geom_test_p90_m'],
    },
    {
        'method': 'plain_gcc',
        'iid_median_m': stable_summary['analytic_plain_gcc']['iid_test']['median_m'],
        'iid_p90_m': stable_summary['analytic_plain_gcc']['iid_test']['p90_m'],
        'geom_median_m': stable_summary['analytic_plain_gcc']['geom_test']['median_m'],
        'geom_p90_m': stable_summary['analytic_plain_gcc']['geom_test']['p90_m'],
    },
    {
        'method': 'gcc_phat',
        'iid_median_m': stable_summary['analytic_gcc_phat']['iid_test']['median_m'],
        'iid_p90_m': stable_summary['analytic_gcc_phat']['iid_test']['p90_m'],
        'geom_median_m': stable_summary['analytic_gcc_phat']['geom_test']['median_m'],
        'geom_p90_m': stable_summary['analytic_gcc_phat']['geom_test']['p90_m'],
    },
]
display(pd.DataFrame(comparison_rows))

analytic_rows = list(csv.DictReader((RESULT_DIR / 'analytic_sample_errors.csv').open('r', encoding='utf-8', newline='')))
analytic_df = pd.DataFrame(analytic_rows)
analytic_df['sample_index'] = analytic_df['sample_index'].astype(int)
analytic_df['position_error_m'] = analytic_df['position_error_m'].astype(float)

def _analytic_pred(sample_index: int, method: str):
    if method == 'oracle':
        lag = true_tdoa[sample_index]
    elif method == 'gcc_phat':
        lag = estimate_tdoa_from_gcc_triplet(gcc_phat[sample_index])
    else:
        raise KeyError(method)
    return estimate_source_position_from_tdoa(lag, ref_positions[sample_index], tuple(cfg['plane_room_size']), int(cfg['fs']), float(cfg['c']))

def plot_localization_sample(sample_index: int, title: str):
    gcc_pred = _analytic_pred(sample_index, 'gcc_phat')
    oracle_pred = _analytic_pred(sample_index, 'oracle')
    tri = ref_positions[sample_index]
    src = source_position[sample_index]
    area = geom_metrics[sample_index, geom_name_to_idx['triangle_area']]
    cond = geom_metrics[sample_index, geom_name_to_idx['jacobian_condition']]
    fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
    ax.scatter(tri[:, 0], tri[:, 1], s=90, color='tab:green', label='refs')
    ax.scatter(src[0], src[1], s=100, color='tab:blue', label='true')
    ax.scatter(gcc_pred[0], gcc_pred[1], s=100, color='tab:orange', label='gcc_phat')
    ax.scatter(oracle_pred[0], oracle_pred[1], s=90, color='tab:red', marker='x', label='oracle')
    ax.set_xlim(0.0, cfg['plane_room_size'][0])
    ax.set_ylim(0.0, cfg['plane_room_size'][1])
    ax.set_title(f"{title} | idx={sample_index} | area={area:.3f} | cond={cond:.2f}")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

subset = analytic_df[(analytic_df['method'] == 'gcc_phat') & (analytic_df['split'] == 'iid_test')].sort_values('position_error_m')
success_idx = int(subset.iloc[0]['sample_index'])
failure_idx = int(subset.iloc[-1]['sample_index'])
plot_localization_sample(success_idx, 'Analytic success')
plot_localization_sample(failure_idx, 'Analytic failure')


In [ ]:
sample_index = int(SAMPLE_INDEX)
if sample_index < 0:
    sample_index = -1
    for idx in range(min(1024, len(true_tdoa))):
        sols = find_tdoa_room_solutions(true_tdoa[idx], ref_positions[idx], tuple(cfg['plane_room_size']), int(cfg['fs']), float(cfg['c']))
        if sols.shape[0] > 1:
            sample_index = idx
            break
    if sample_index < 0:
        raise RuntimeError('No multi-solution sample found in the scanned subset.')

solutions = find_tdoa_room_solutions(true_tdoa[sample_index], ref_positions[sample_index], tuple(cfg['plane_room_size']), int(cfg['fs']), float(cfg['c']))
tri = ref_positions[sample_index]
src = source_position[sample_index]
obs_rms = np.sqrt(np.mean(np.square(x_ref[sample_index]), axis=1) + 1.0e-8)
obs_log_rms_abs = np.log(obs_rms + 1.0e-8)
obs_log_rms_center = obs_log_rms_abs - np.mean(obs_log_rms_abs)

energy_rows = []
for ridx, sol in enumerate(solutions):
    dist = np.linalg.norm(sol[None, :] - tri, axis=1)
    proxy_abs = -np.log(dist + 1.0e-6)
    proxy_center = proxy_abs - np.mean(proxy_abs)
    energy_rows.append({
        'root_id': ridx,
        'x': float(sol[0]),
        'y': float(sol[1]),
        'abs_mse': float(np.mean((obs_log_rms_abs - proxy_abs) ** 2)),
        'center_mse': float(np.mean((obs_log_rms_center - proxy_center) ** 2)),
        'mean_proxy_abs': float(np.mean(proxy_abs)),
    })
energy_df = pd.DataFrame(energy_rows).sort_values(['abs_mse', 'center_mse'])
display(energy_df)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.4), constrained_layout=True)
axes[0].scatter(tri[:, 0], tri[:, 1], s=90, color='tab:green', label='refs')
axes[0].scatter(src[0], src[1], s=110, color='tab:blue', label='true source')
if len(solutions):
    axes[0].scatter(solutions[:, 0], solutions[:, 1], s=90, color='tab:red', marker='x', label='TDOA-only roots')
for ridx, sol in enumerate(solutions):
    axes[0].text(sol[0] + 0.03, sol[1] + 0.03, f'r{ridx}', fontsize=10)
area = geom_metrics[sample_index, geom_name_to_idx['triangle_area']]
cond = geom_metrics[sample_index, geom_name_to_idx['jacobian_condition']]
axes[0].set_xlim(0.0, cfg['plane_room_size'][0])
axes[0].set_ylim(0.0, cfg['plane_room_size'][1])
axes[0].grid(True, alpha=0.25)
axes[0].legend()
axes[0].set_title(f"TDOA-only roots | idx={sample_index} | area={area:.3f} | cond={cond:.2f}")

axes[1].plot(np.arange(3), obs_log_rms_abs, '-o', linewidth=2.0, label='observed log RMS (abs)')
for ridx, sol in enumerate(solutions[:4]):
    dist = np.linalg.norm(sol[None, :] - tri, axis=1)
    proxy_abs = -np.log(dist + 1.0e-6)
    axes[1].plot(np.arange(3), proxy_abs, '--o', alpha=0.85, label=f'root {ridx} proxy')
axes[1].set_xticks(np.arange(3), labels=['ref0', 'ref1', 'ref2'])
axes[1].set_ylabel('log-amplitude / proxy')
axes[1].set_title('Absolute energy can help pick the right root')
axes[1].grid(True, alpha=0.25)
axes[1].legend(loc='best')
plt.show()


In [ ]:
result_specs = [
    ('Pure MLP', MLP_RESULT_DIR, 'iid_tdoa_energy_canonical_mlp'),
    ('Abs-Feat MLP', ABS_RESULT_DIR, 'iid_tdoa_energy_abs_canonical_mlp'),
    ('Physics MLP', PHYS_RESULT_DIR, 'iid_tdoa_energy_abs_phys_mlp_l025'),
    ('Solver', SOLVER_RESULT_DIR, 'iid_solver_tdoa_energy_anchor'),
]

metric_rows = []
bundles = {}
for label, result_dir, model_key in result_specs:
    if not (result_dir / 'summary.json').exists():
        continue
    summary = load_stable_localization_summary(result_dir)
    learned = summary.get('learned', {})
    if model_key not in learned:
        continue
    bundles[label] = load_stage01_candidate_prediction_bundle(H5_PATH, result_dir, model_key=model_key, device='auto')
    metric_rows.append({
        'label': label,
        'model_key': model_key,
        'median_m': learned[model_key]['median_m'],
        'p90_m': learned[model_key]['p90_m'],
        'mean_m': learned[model_key]['mean_m'],
        'max_m': learned[model_key]['max_m'],
    })

metric_df = pd.DataFrame(metric_rows).sort_values(['p90_m', 'median_m'])
display(metric_df)

if bundles:
    common_sample = min(300, min(bundle['pred_global'].shape[0] for bundle in bundles.values()))
    pick = np.random.default_rng(0).choice(np.arange(common_sample), size=common_sample, replace=False)
    analytic_idx = next(iter(bundles.values()))['indices'][pick]
    analytic_pred = []
    for idx in analytic_idx:
        lag = estimate_tdoa_from_gcc_triplet(gcc_phat[idx])
        analytic_pred.append(estimate_source_position_from_tdoa(lag, ref_positions[idx], tuple(cfg['plane_room_size']), int(cfg['fs']), float(cfg['c'])))
    analytic_pred = np.asarray(analytic_pred, dtype=np.float32)
    analytic_true = source_position[analytic_idx]
    analytic_err = np.linalg.norm(analytic_pred - analytic_true, axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.5), constrained_layout=True)
    axes[0].scatter(analytic_true[:, 0], analytic_true[:, 1], s=14, alpha=0.20, color='black', label='true')
    axes[0].scatter(analytic_pred[:, 0], analytic_pred[:, 1], s=16, alpha=0.45, label='analytic gcc-phat')
    for label, bundle in bundles.items():
        axes[0].scatter(bundle['pred_global'][pick, 0], bundle['pred_global'][pick, 1], s=16, alpha=0.40, label=label)
    axes[0].set_xlim(0.0, cfg['plane_room_size'][0])
    axes[0].set_ylim(0.0, cfg['plane_room_size'][1])
    axes[0].grid(True, alpha=0.25)
    axes[0].set_title('Analytic vs learned predictions on the same stable split subset')
    axes[0].legend(loc='best', fontsize=9)

    axes[1].hist(analytic_err, bins=40, alpha=0.50, label='analytic gcc-phat')
    for label, bundle in bundles.items():
        axes[1].hist(bundle['error_m'], bins=40, alpha=0.45, label=label)
    axes[1].set_xlabel('error_m')
    axes[1].set_ylabel('count')
    axes[1].grid(True, alpha=0.25)
    axes[1].set_title('Pure MLP vs physics loss vs solver')
    axes[1].legend(loc='best', fontsize=9)
    plt.show()
else:
    print('No learned result directories were found.')
